# Sliding-Window Pairwise Electrode Correlation

Compute a time-resolved `n_ch x n_ch` zero-lag Pearson correlation matrix for each of BLT, P1, and P2 (500 ms gap only), on a shared time axis, then compare.

- Signal per condition: trial-averaged evoked.
- Window: `WINDOW_MS` ms, step `STEP_MS` ms.
- Output per condition: `(n_windows, n_ch, n_ch)` + window-center times.

## Setup & config

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.stats.correlation_tools as correlation_tools
import statsmodels.tsa.stattools as stattools
import scipy.stats as stats
#from scikit_rmt.ensemble.spectral_law import MarchenkoPasturDistribution

from utils import load_eeg_data
from utils import plot_all_channels_by_trial
from connectivity import (
    select_p2_500ms,
    sliding_corr_from_epochs,
    summary_metrics,
    lagged_corr_2d,
    sliding_lagged_diff_corr_from_epochs
)

SUBJECT = 5
TMIN, TMAX = None, None       # native epoch window (must match across files)
WINDOW_MS = 200
STEP_MS = 50
FREQ_BAND = None              # e.g. 'alpha' / 'beta' for Cell G
LAGS_SEC = np.linspace(-0.05, 0.05, 81)

## Load conditions on a common time axis

In [ ]:
blt = load_eeg_data('BLT', SUBJECT, tmin=TMIN, tmax=TMAX, freq_band=FREQ_BAND, normalize=True)
bla = load_eeg_data('BLA', SUBJECT, tmin=TMIN, tmax=TMAX, freq_band=FREQ_BAND, normalize=True)
p1_first_10  = load_eeg_data('P1',  SUBJECT, trials = list(range(10)) , tmin=TMIN, tmax=TMAX, freq_band=FREQ_BAND, normalize=True)
p1_last_10 = load_eeg_data('P1',  SUBJECT, trials = list(range(-10,0)) , tmin=TMIN, tmax=TMAX, freq_band=FREQ_BAND, normalize=True)
p1 = load_eeg_data('P1',  SUBJECT, tmin=TMIN, tmax=TMAX, freq_band=FREQ_BAND, normalize=True)
p2_all = load_eeg_data('P2', SUBJECT, tmin=TMIN, tmax=TMAX, freq_band=FREQ_BAND, normalize=True)
p2 = select_p2_500ms(p2_all)
p2_first_10 = load_eeg_data('P2',  SUBJECT, trials = list(range(10)) , tmin=TMIN, tmax=TMAX, freq_band=FREQ_BAND, normalize=True)
p2_last_10 = load_eeg_data('P2',  SUBJECT, trials = list(range(-10,0)) , tmin=TMIN, tmax=TMAX, freq_band=FREQ_BAND, normalize=True)
p2_500_first_10 = select_p2_500ms(p2_first_10)
p2_500_last_10 = select_p2_500ms(p2_last_10)



assert np.allclose(blt.times, p1.times), 'BLT and P1 time axes differ'
assert np.allclose(p1.times, p2.times), 'P1 and P2 time axes differ'

# Channel-set alignment: intersect across files so the NxN matrices line up.
common_ch = [c for c in blt.ch_names if c in p1.ch_names and c in p2.ch_names]
for ep in (blt, p1, p2):
    ep.pick_channels(common_ch, ordered=True)

    conds = {'BLT': blt, 'P1': p1, "P1 First 10": p1_first_10, "P1 Last 10": p1_last_10,
             'P2 First 10': p2_first_10, 'P2 Last 10': p2_last_10,"BLT": blt, "BLA": bla, "P2 All": p2_all, "P2": p2, "P2 500 First 10": p2_500_first_10, "P2 500 Last 10": p2_500_last_10}
    results = {}
    for name, ep in conds.items():
        diff_corr, raw_corr, centers, ch_names = sliding_lagged_diff_corr_from_epochs(
            ep, WINDOW_MS, STEP_MS, LAGS_SEC
        )
        results[name] = {
            'corr':     diff_corr,
            'raw_corr': raw_corr,
            'centers':  centers,
            'ch_names': ch_names,
        }

cond_names = list(results.keys())

all_data = np.concatenate([results[n]['corr'] for n in cond_names])
clim     = np.nanpercentile(np.abs(all_data), 99)

clear_output()

print(f'n_channels (common): {len(common_ch)}')
print(f'n_times: {len(blt.times)}   sfreq: {blt.info["sfreq"]} Hz')
print(f'trials BLT/P1/P2_500ms: {len(blt)}/{len(p1)}/{len(p2)}')

## Time-course summary per condition

`mean_abs_r` = average `|r|` across unique channel pairs; `density(0.5)` = fraction of pairs with `|r| > 0.5`.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
for name, res in results.items():
    m = summary_metrics(res['corr'])
    axes[0].plot(res['centers'], m['mean_abs_r'], label=name)
    axes[1].plot(res['centers'], m['density'], label=name)
axes[0].set_ylabel('mean |r|')
axes[1].set_ylabel('density (|r|>0.5)')
axes[1].set_xlabel('time (s)')
for ax in axes:
    ax.axvline(0, color='k', lw=0.5, alpha=0.4)
    ax.legend(loc='best', fontsize=9)
fig.suptitle(f'Sliding-window connectivity (window={WINDOW_MS} ms, step={STEP_MS} ms)')
plt.tight_layout()
plt.show()

## Heatmap grid at key time points

Pick a few window indices and show the full `n_ch x n_ch` matrix for each condition.

In [ ]:
centers = results['BLT']['centers']
n_windows = len(centers)
# Pick 5 evenly spaced windows across the epoch.
pick_idx = np.linspace(0, n_windows - 1, 5, dtype=int)
pick_times = centers[pick_idx]

cond_names = list(results.keys())
fig, axes = plt.subplots(
    len(cond_names), len(pick_idx),
    figsize=(3.0 * len(pick_idx), 3.0 * len(cond_names)),
    squeeze=False,
)
for i, name in enumerate(cond_names):
    corr = results[name]['corr']
    for j, w in enumerate(pick_idx):
        ax = axes[i, j]
        im = ax.imshow(corr[w], vmin=-1, vmax=1, cmap='RdBu_r', aspect='auto')
        ax.set_xticks([])
        ax.set_yticks([])
        if i == 0:
            ax.set_title(f't={pick_times[j]:+.2f}s', fontsize=10)
        if j == 0:
            ax.set_ylabel(name, fontsize=11)
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.6, label='Pearson r')
fig.suptitle('Correlation matrices at selected windows')
plt.show()

## Condition-difference heatmaps

P1 - BLT isolates the effect of adding the auditory cue (with a fixed 500 ms gap). P2_500ms - P1 isolates paradigm/block differences when the audio->tactile interval is matched.

In [ ]:
diffs = {
    'P1 - BLT': results['P1']['corr'] - results['BLT']['corr'],
    'P2_500ms - P1': results['P2_500ms']['corr'] - results['P1']['corr'],
}
# Symmetric color range per diff for fair reading.
fig, axes = plt.subplots(
    len(diffs), len(pick_idx),
    figsize=(3.0 * len(pick_idx), 3.0 * len(diffs)),
    squeeze=False,
)
for i, (name, d) in enumerate(diffs.items()):
    vmax = np.nanpercentile(np.abs(d), 99)
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0
    for j, w in enumerate(pick_idx):
        ax = axes[i, j]
        im = ax.imshow(d[w], vmin=-vmax, vmax=vmax, cmap='RdBu_r', aspect='auto')
        ax.set_xticks([])
        ax.set_yticks([])
        if i == 0:
            ax.set_title(f't={pick_times[j]:+.2f}s', fontsize=10)
        if j == 0:
            ax.set_ylabel(name, fontsize=11)
    fig.colorbar(im, ax=axes[i, :].tolist(), shrink=0.7, label=f'\u0394r (vmax={vmax:.2f})')
fig.suptitle('Condition differences in correlation structure')
plt.show()

## Per-band repeat

Re-run B-F with `FREQ_BAND = 'alpha'` or `'beta'` to check whether the dynamics are band-specific.

In [ ]:
frequencies = ['alpha', 'beta']
for freq in frequencies:
    blt = load_eeg_data('BLT', SUBJECT, tmin=TMIN, tmax=TMAX, freq_band = freq, normalize=True)
    p1  = load_eeg_data('P1',  SUBJECT, tmin=TMIN, tmax=TMAX, freq_band=freq, normalize=True)
    p2_all = load_eeg_data('P2', SUBJECT, tmin=TMIN, tmax=TMAX, freq_band=freq, normalize=True)
    p2 = select_p2_500ms(p2_all)
    
    assert np.allclose(blt.times, p1.times), 'BLT and P1 time axes differ'
    assert np.allclose(p1.times, p2.times), 'P1 and P2 time axes differ'
    
    # Channel-set alignment: intersect across files so the NxN matrices line up.
    common_ch = [c for c in blt.ch_names if c in p1.ch_names and c in p2.ch_names]
    for ep in (blt, p1, p2):
        ep.pick_channels(common_ch, ordered=True)
        
    centers = results['BLT']['centers']
    n_windows = len(centers)
    # Pick 5 evenly spaced windows across the epoch.
    pick_idx = np.linspace(0, n_windows - 1, 5, dtype=int)
    pick_times = centers[pick_idx]
    
    cond_names = list(results.keys())
    fig, axes = plt.subplots(
        len(cond_names), len(pick_idx),
        figsize=(3.0 * len(pick_idx), 3.0 * len(cond_names)),
        squeeze=False,
    )
    for i, name in enumerate(cond_names):
        corr = results[name]['corr']
        for j, w in enumerate(pick_idx):
            ax = axes[i, j]
            im = ax.imshow(corr[w], vmin=-1, vmax=1, cmap='RdBu_r', aspect='auto')
            ax.set_xticks([])
            ax.set_yticks([])
            if i == 0:
                ax.set_title(f't={pick_times[j]:+.2f}s', fontsize=10)
            if j == 0:
                ax.set_ylabel(name, fontsize=11)
    fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.6, label='Pearson r')
    fig.suptitle('Correlation matrices at selected windows')
    plt.show()
    
    diffs = {
        'P1 - BLT': results['P1']['corr'] - results['BLT']['corr'],
        'P2_500ms - P1': results['P2_500ms']['corr'] - results['P1']['corr'],
    }
    # Symmetric color range per diff for fair reading.
    fig, axes = plt.subplots(
        len(diffs), len(pick_idx),
        figsize=(3.0 * len(pick_idx), 3.0 * len(diffs)),
        squeeze=False,
    )
    for i, (name, d) in enumerate(diffs.items()):
        vmax = np.nanpercentile(np.abs(d), 99)
        if not np.isfinite(vmax) or vmax == 0:
            vmax = 1.0
        for j, w in enumerate(pick_idx):
            ax = axes[i, j]
            im = ax.imshow(d[w], vmin=-vmax, vmax=vmax, cmap='RdBu_r', aspect='auto')
            ax.set_xticks([])
            ax.set_yticks([])
            if i == 0:
                ax.set_title(f't={pick_times[j]:+.2f}s', fontsize=10)
            if j == 0:
                ax.set_ylabel(name, fontsize=11)
        fig.colorbar(im, ax=axes[i, :].tolist(), shrink=0.7, label=f'\u0394r (vmax={vmax:.2f})')
    fig.suptitle('Condition differences in correlation structure')
    plt.show()
    

## Animation of the Correlation as a Function of Time(BLT)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import io
from PIL import Image

In [ ]:
centers = results['BLT']['centers']
cond_names = list(results.keys())

frame_images = []

for w in range(len(centers)):
    fig, axes = plt.subplots(
        len(cond_names), 1,
        figsize=(3.5, 3.0 * len(cond_names)),
        squeeze=False,
        constrained_layout=True,   # ← replaces tight_layout
    )

    for i, name in enumerate(cond_names):
        corr = results[name]['corr']
        ax = axes[i, 0]
        im = ax.imshow(corr[w], vmin=-1, vmax=1, cmap='RdBu_r', aspect='auto')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_ylabel(name, fontsize=11)

    fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.6, label='Pearson r')
    fig.suptitle(f'Correlation matrices — t={centers[w]:+.2f}s', fontsize=12)
    # NO plt.tight_layout() here

    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100)
    buf.seek(0)
    frame_images.append(np.array(Image.open(buf)))
    plt.close(fig)

import matplotlib.animation as animation
from IPython.display import HTML
from matplotlib import rc
import matplotlib
matplotlib.rcParams['animation.embed_limit'] = 2**128
rc('animation', html='jshtml')

fig, ax = plt.subplots(figsize=(12, 12))
ax.axis("off")
im = ax.imshow(frame_images[0])

def update(frame):
    im.set_array(frame_images[frame])
    return im,

ani = animation.FuncAnimation(
    fig,
    update,
    frames=len(frame_images),
    interval=200,
    blit=True,
    repeat=True
)

plt.close(fig)
HTML(ani.to_jshtml())

## Test for Linear Lag

observe difference in correlation between lagged/unlagged, if you see uniform increase in correlation, should hint toward linear lag

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

%matplotlib inline

def redraw(epoch: str, change=None):
    diff_corr = results[epoch]['corr']      # (n_windows, n_lags, n_ch, n_ch)
    raw_corr  = results[epoch]['raw_corr']  # (n_windows, n_lags, n_ch, n_ch)
    ch_names  = results[epoch]['ch_names']

    lag_idx = lag_slider.value
    lag     = LAGS_SEC[lag_idx]

    # ── Global optimal lag ────────────────────────────────────────────────────
    temporal_variance   = raw_corr.var(axis=0)            # (n_lags, n_ch, n_ch)
    global_variance     = temporal_variance.mean(axis=(1, 2))  # (n_lags,)
    best_lag_global_idx = global_variance.argmin()
    best_lag_global_sec = LAGS_SEC[best_lag_global_idx]

    # ── Per-pair optimal lag ──────────────────────────────────────────────────
    best_lag_idx_pp  = temporal_variance.argmin(axis=0)   # (n_ch, n_ch)
    best_lag_sec_pp  = LAGS_SEC[best_lag_idx_pp]

    raw_corr_mean    = raw_corr.mean(axis=0)              # (n_lags, n_ch, n_ch)
    best_lag_corr_pp = np.take_along_axis(
        raw_corr_mean, best_lag_idx_pp[np.newaxis], axis=0
    ).squeeze(0)
    best_lag_var_pp  = np.take_along_axis(
        temporal_variance, best_lag_idx_pp[np.newaxis], axis=0
    ).squeeze(0)

    threshold     = np.percentile(best_lag_var_pp, 50)
    graph_weights = np.where(best_lag_var_pp < threshold, best_lag_corr_pp, 0)

    # Time-averaged correlation at global optimal lag
    corr_at_global = raw_corr_mean[best_lag_global_idx]   # (n_ch, n_ch)

    # ── Nearest correlation matrix (Higham) ───────────────────────────────────
    def symmetrize_matrix(A):
        return (A + A.T) / 2

    def project_to_positive_semidefinite(A):
        eigenvalues, eigenvectors = np.linalg.eigh(A)
        A_psd = (eigenvectors * np.maximum(eigenvalues, 0)).dot(eigenvectors.T)
        return symmetrize_matrix(A_psd)

    def nearest_correlation_matrix(A, tol=1e-8, max_iterations=100):
        X = symmetrize_matrix(A)
        correction_matrix = np.zeros_like(X)
        for _ in range(max_iterations):
            X_old = X.copy()
            residual = X - correction_matrix
            X = project_to_positive_semidefinite(residual)
            correction_matrix = X - residual
            np.fill_diagonal(X, 1)
            if np.linalg.norm(X - X_old, 'fro') / np.linalg.norm(X, 'fro') < tol:
                break
        return X

    higham_corr = nearest_correlation_matrix(best_lag_corr_pp)

    lambda_max = 4
    x          = np.linspace(.01, lambda_max, 500)
    def mpdist(lambd):
        return np.sqrt((lambda_max - lambd) * lambd) / (2 * np.pi * lambd)

    raw_eig           = np.linalg.eig(corr_at_global)
    normal_eig        = np.linalg.eig(best_lag_corr_pp)
    lag_corrected_eig = np.linalg.eig(higham_corr)
    corrected_corr    = correlation_tools.corr_clipped(corr_at_global, threshold=lambda_max)

    # ── Plot ──────────────────────────────────────────────────────────────────
    with out:
        clear_output(wait=True)
        fig, axes = plt.subplots(3, 3, figsize=(18, 12), constrained_layout=True)

        # [0,0] Δ Corr averaged over all time at selected lag
        mean_diff_at_lag = results[epoch]['corr'][:, lag_idx].mean(axis=0)
        im0 = axes[0, 0].imshow(mean_diff_at_lag, vmin=-clim, vmax=clim,
                                 cmap='RdBu_r', aspect='equal')
        axes[0, 0].set_title(f'Δ Corr (time-avg)  |  lag={lag:+.3f}s')
        fig.colorbar(im0, ax=axes[0, 0], label='Δ Pearson r', shrink=0.8)

        # [0,1] Variance curve across lags
        axes[0, 1].plot(LAGS_SEC * 1000, global_variance, color='steelblue')
        axes[0, 1].axvline(best_lag_global_sec * 1000, color='r', linestyle='--',
                           label=f'optimal = {best_lag_global_sec*1000:+.1f}ms')
        axes[0, 1].axvline(lag * 1000, color='orange', linestyle=':',
                           label=f'selected = {lag*1000:+.1f}ms')
        axes[0, 1].set_xlabel('Lag (ms)')
        axes[0, 1].set_ylabel('Mean temporal variance')
        axes[0, 1].set_title('Stability across lags (global)')
        axes[0, 1].legend(fontsize=9)

        # [0,2] Correlation at global optimal lag (time-averaged)
        vmax_g = np.abs(corr_at_global).max()
        im2 = axes[0, 2].imshow(corr_at_global, cmap='RdBu_r', aspect='equal',
                                  vmin=-vmax_g, vmax=vmax_g)
        axes[0, 2].set_title(f'Corr at global optimal lag ({best_lag_global_sec*1000:+.1f}ms, time-avg)')
        fig.colorbar(im2, ax=axes[0, 2], label='Pearson r', shrink=0.8)

        # [1,0] Per-pair optimal lag map
        im3 = axes[1, 0].imshow(best_lag_sec_pp, cmap='RdBu_r', aspect='equal',
                                  vmin=-LAGS_SEC.max(), vmax=LAGS_SEC.max())
        axes[1, 0].set_title('Per-pair optimal lag (s)')
        fig.colorbar(im3, ax=axes[1, 0], label='lag (s)', shrink=0.8)

        # [1,1] Correlation at per-pair optimal lag (time-averaged)
        vmax_pp = np.abs(best_lag_corr_pp).max()
        im4 = axes[1, 1].imshow(best_lag_corr_pp, cmap='RdBu_r', aspect='equal',
                                  vmin=-vmax_pp, vmax=vmax_pp)
        axes[1, 1].set_title('Corr at per-pair optimal lag (time-avg)')
        fig.colorbar(im4, ax=axes[1, 1], label='Pearson r', shrink=0.8)

        # [1,2] Graph weights (thresholded)
        vmax_w = np.abs(graph_weights).max() or 1
        im5 = axes[1, 2].imshow(graph_weights, cmap='RdBu_r', aspect='equal',
                                  vmin=-vmax_w, vmax=vmax_w)
        axes[1, 2].set_title('Graph weights (thresholded to 50th percentile)')
        fig.colorbar(im5, ax=axes[1, 2], label='Pearson r', shrink=0.8)

        # [2,0] Eigenvalue distributions
        axes[2, 0].hist(normal_eig[0],        bins=25, density=True, label='Lag-corrected')
        axes[2, 0].hist(lag_corrected_eig[0], bins=25, density=True, label='Higham')
        axes[2, 0].hist(raw_eig[0],           bins=25, density=True, label='Raw')
        axes[2, 0].plot(x, mpdist(x), label='MP distribution')
        axes[2, 0].legend()
        axes[2, 0].set_title('Eigenvalue distributions')

        # [2,1] Corrected graph weights
        vmax_w = np.abs(corrected_corr).max() or 1
        im6 = axes[2, 1].imshow(corrected_corr, cmap='RdBu_r', aspect='equal',
                                  vmin=-vmax_w, vmax=vmax_w)
        axes[2, 1].set_title('Corrected Graph Weights(without Higham)')
        fig.colorbar(im6, ax=axes[2, 1], label='Pearson r', shrink=0.8)

        # [2,2] Higham correlation matrix
        vmax_h = np.abs(higham_corr).max()
        im7 = axes[2, 2].imshow(higham_corr, cmap='RdBu_r', aspect='equal',
                                  vmin=-vmax_h, vmax=vmax_h)
        axes[2, 2].set_title("Correlation matrix (Higham's Algorithm)")
        fig.colorbar(im7, ax=axes[2, 2], label='Pearson r', shrink=0.8)

        for ax in [axes[0, 0], axes[0, 2], axes[1, 0], axes[1, 1], axes[1, 2]]:
            ax.set_xticks(range(len(ch_names)))
            ax.set_yticks(range(len(ch_names)))
            ax.set_xticklabels(ch_names, rotation=90, fontsize=6)
            ax.set_yticklabels(ch_names, fontsize=6)

        fig.suptitle(f'Time-averaged  |  lag = {lag:+.3f}s  |  epoch = {epoch}', fontsize=12)
        display(fig)
        plt.close(fig)

    return corrected_corr, higham_corr, corr_at_global, best_lag_sec_pp, best_lag_corr_pp


lag_slider = widgets.SelectionSlider(
    options=[(f'{l:+.3f}s', i) for i, l in enumerate(LAGS_SEC)],
    description='Lag:',
    layout=widgets.Layout(width='700px'),
)

cond_dropdown = widgets.Dropdown(
    options=cond_names,
    value=cond_names[0],
    description='Condition:',
    layout=widgets.Layout(width='300px'),
)

out = widgets.Output()

def on_change(change=None):
    redraw(cond_dropdown.value)

lag_slider.observe(on_change, names='value')
cond_dropdown.observe(on_change, names='value')

display(widgets.HBox([cond_dropdown, lag_slider]), out)
corrected, higham, glob_corr, lags, lagged_corrs = redraw(cond_names[0])

#lag_slider.observe(lambda change: redraw('P1', change), names='value')
#display(lag_slider, out)
#corrected, higham, glob_corr, lags, lagged_corrs= redraw('P1')

NameError: name 'LAGS_SEC' is not defined

## Experiments using an anchor node

In [ ]:
from lagged_graph import (
    sliding_lagged_from_epochs,
    build_directed_graph,
    draw_lagged_graph,
    scalp_positions,
    consistency_across_windows,
    anchor_sliding_lagged_from_epochs,
    anchor_pair_sliding_curves_from_epochs,
    draw_sliding_lag_heatmaps,
)

from connectivity import (
    select_p2_500ms,
    roi_order,
    reorder_corr,
    draw_roi_heatmap,
    draw_roi_anchor_timeseries,
)

anchor = 'C3'

a_r, a_lag, a_r0, a_centers, _ = anchor_sliding_lagged_from_epochs(
    epochs,
    anchors=[anchor],
    window_ms=WINDOW_MS,
    step_ms=STEP_MS,
    max_lag_ms=MAX_LAG_MS,
    use='trial_average',
)

draw_roi_anchor_timeseries(a_r, a_centers, ch_names, anchor='C3')
plt.show()

## Graph implementation


## Julia Code
The below code was all run in Julia, and produced the csvs used later(there was other processing code, but it was unwieldy to put it all in a python notebook since it wouldn't run anyways, the files and input/output data for the code will also be submitted in case)

The full documentation for our process here is in our final report:

```
using LoopVectorization
using LinearAlgebra


using Statistics
using JLD2
using GLMakie
using CSV
using DataFrames


function get_covariance(M::Matrix{Float64}, lag::Matrix{Integer}, m_size::Integer = 32)
    local Σ::Matrix{Float64} = zeros(m_size, m_size)
    @inbounds for i ∈ 1:m_size
        for j ∈ 1:m_size
            offset = lag[i, j]
            if offset == 0
                Σ[i, j] = cov(M[:, i], M[:, j])
            elseif offset > 0
                Σ[i, j] = cov(M[1 + offset:end, i], M[1:end - offset, j])
            elseif offset < 0
                Σ[i, j] = cov(M[1:end + offset, i], M[1 - offset:end, j])
            end
        end
    end
    return Σ
end


function nearest_psd(Σ::Matrix{Float64})
    Σ_symmetric::Matrix{Float64} = UpperTriangular(Σ)' + UpperTriangular(Σ) - Diagonal(Σ)
    decomposition = eigen(Σ_symmetric)
    λ_vec::Vector{Float64} = decomposition.values
    v_mat::Matrix{Float64} = decomposition.vectors
    return Symmetric(v_mat * Diagonal(max.(0.0, λ_vec)) * v_mat^-1)
end


function covariance_to_correlation(Σ::Symmetric{Float64, Matrix{Float64}})
    diag_factor::Matrix{Float64} = sqrt.(Diagonal(Σ))^-1
    return Symmetric(diag_factor * Σ * diag_factor)
end


function apply_threshold(Σ::Symmetric{Float64, Matrix{Float64}}, R::Symmetric{Float64, Matrix{Float64}}, threshold::Float64)
    R_values::Vector{Float64} = eigen(R).values
    decomposition = eigen(Σ)
    Σ_values::Vector{Float64} = decomposition.values
    Σ_vectors::Matrix{Float64} = decomposition.vectors
    return Symmetric(Σ_vectors * Diagonal(map(i -> if R_values[i] > threshold return Σ_values[i] else return 0.0 end, 1:32)) * Σ_vectors^-1)
end


t_samples::Integer = 1792
#normalization_factor::Integer = 2 * t_samples^2
threshold::Float64 = (1.0 + sqrt(32 / 1770))^2


sdat = load(normpath(joinpath((@__FILE__), raw"..\subject5dat.jld2")))
lag_matrix::Matrix{Integer} = sdat["lag_matrix"]
lag_matrix2::Matrix{Integer} = Matrix{Integer}(CSV.read(normpath(joinpath((@__FILE__), raw"..\lagindices2.csv")), DataFrame))


buffer::Matrix{Float64} = zeros(32, 32) 
covariance::Symmetric{Float64, Matrix{Float64}} = Symmetric(zeros(32, 32))
correlation::Symmetric{Float64, Matrix{Float64}} = Symmetric(zeros(32, 32))
new_covariance::Symmetric{Float64, Matrix{Float64}} = Symmetric(zeros(32, 32))
@inbounds for i ∈ 1:10
    covariance = nearest_psd(get_covariance(sdat["CISI_data"][2i], lag_matrix))
    correlation = covariance_to_correlation(covariance)
    new_covariance = apply_threshold(covariance, correlation, threshold)
    buffer =+ new_covariance
end
buffer ./= 10
cisi_first10_avg_correlation = covariance_to_correlation(nearest_psd(buffer))
buffer = zeros(32, 32)
covariance = Symmetric(zeros(32, 32))
correlation = Symmetric(zeros(32, 32))
new_covariance = Symmetric(zeros(32, 32))
@inbounds for i ∈ 51:60
    covariance = nearest_psd(get_covariance(sdat["CISI_data"][2i], lag_matrix))
    correlation = covariance_to_correlation(covariance)
    new_covariance = apply_threshold(covariance, correlation, threshold)
    buffer =+ new_covariance
end
buffer ./= 10
cisi_last10_avg_correlation = covariance_to_correlation(nearest_psd(buffer))


buffer = zeros(32, 32)
covariance = Symmetric(zeros(32, 32))
correlation = Symmetric(zeros(32, 32))
new_covariance = Symmetric(zeros(32, 32))
@inbounds for i ∈ 1:10
    covariance = nearest_psd(get_covariance(sdat["VISI_data"][2i], lag_matrix2))
    correlation = covariance_to_correlation(covariance)
    new_covariance = apply_threshold(covariance, correlation, threshold)
    buffer =+ new_covariance
end
buffer ./= 10
visi_first10_avg_correlation = covariance_to_correlation(nearest_psd(buffer))
buffer = zeros(32, 32)
covariance = Symmetric(zeros(32, 32))
correlation = Symmetric(zeros(32, 32))
new_covariance = Symmetric(zeros(32, 32))
@inbounds for i ∈ 51:60
    covariance = nearest_psd(get_covariance(sdat["VISI_data"][2i], lag_matrix2))
    correlation = covariance_to_correlation(covariance)
    new_covariance = apply_threshold(covariance, correlation, threshold)
    buffer =+ new_covariance
end
buffer ./= 10
visi_last10_avg_correlation = covariance_to_correlation(nearest_psd(buffer))


df = DataFrame(visi_first10_avg_correlation, :auto)
CSV.write(normpath(joinpath((@__FILE__), raw"..\visifirst10.csv")), df)
df = DataFrame(visi_last10_avg_correlation, :auto)
CSV.write(normpath(joinpath((@__FILE__), raw"..\visilast10.csv")), df)


fig = Figure(size = (1600, 900))
ax = Axis(fig[1, 1], title = "Change in Post-Processed Correlation Matrix Between CISI and VISI (last ten CISI trials vs first ten VISI trials)", ylabel = "Lagged Index", xlabel = "Non-Lagged Index", aspect = DataAspect(),
    xticks = collect(1:32), yticks = collect(1:32), titlesize = 24, xlabelsize = 24, ylabelsize = 24
)
hm = heatmap!(ax, (1, 32), (1, 32), visi_first10_avg_correlation - cisi_last10_avg_correlation, colormap = Reverse(:seismic), interpolate = false) #colorscale = ReversibleScale(x -> sign(x) * sqrt(abs(x)), x -> sign(x) * x^2),
Colorbar(fig[:, end + 1], hm, ticks = -1:0.1:1, label = "Change in Correlation", labelsize = 24)
save(normpath(joinpath((@__FILE__), raw"..\cisitovisichange.png")), fig)
fig
```

In [ ]:
import plotly.graph_objects as go
def plot_3d_graph(g, seed = None):
        pos = nx.forceatlas2_layout(g, dim=3, seed = seed)

        edge_x, edge_y, edge_z = [], [], []
        for u, v in g.edges():
            x0, y0, z0 = pos[u]
            x1, y1, z1 = pos[v]
            edge_x += [x0, x1, None]
            edge_y += [y0, y1, None]
            edge_z += [z0, z1, None]

        edge_trace = go.Scatter3d(
            x=edge_x, y=edge_y, z=edge_z,
            mode='lines',
            line=dict(color='grey', width=1),
            hoverinfo='none'
        )

        node_traces = []
        for group, color in colordict.items():
            nodes_in_group = [n for n in g.nodes() if g.nodes[n]['group'] == group]
            if not nodes_in_group:
                continue
            node_traces.append(go.Scatter3d(
                x=[pos[n][0] for n in nodes_in_group],
                y=[pos[n][1] for n in nodes_in_group],
                z=[pos[n][2] for n in nodes_in_group],
                mode='markers+text',
                name=group,
                text=[g.nodes[n]['name'] for n in nodes_in_group],
                textposition='top center',
                marker=dict(size=6, color=color),
                hoverinfo='text'
            ))

        fig = go.Figure(data=[edge_trace, *node_traces])
        fig.update_layout(
            showlegend=True,
            scene=dict(
                xaxis=dict(showbackground=False),
                yaxis=dict(showbackground=False),
                zaxis=dict(showbackground=False)
            ),
            margin=dict(l=0, r=0, t=0, b=0)
        )
        fig.show()

def delete_node(g, node_names):
    indices = [n for n in g.nodes if g.nodes[n]['name'] in node_names]
    g_deleted = g.copy()
    for n in indices:
        g_deleted.remove_node(n)
        print(f"Deleted node: {n}")
    return g_deleted
    
        
#plot_3d_graph(g_signal_noise)


In [ ]:
import networkx as nx
import numpy as np

frontal = ['FPZ', 'AF7', 'AF8', 'AF3', 'AF4', 'F3', 'FZ', 'F4']
frontocentral = ['FC5', 'FC1', 'FC2', 'FC6']
somatosensory = ['C3', 'Cz', 'C4']
centroparietal = ['CP3', 'CP1', 'CPZ', 'CP2', 'CP4']
temporal = ['T7', 'T8', 'TP7', 'TP8']
parietal_occipital = ['P5', 'P1', 'P2', 'P6', 'POz', 'O1', 'Oz', 'O2']

groups = {"frontal": frontal, "frontocentral": frontocentral, "somatosensory": somatosensory,
          "centroparietal": centroparietal, "temporal": temporal, "parietal_occipital": parietal_occipital}
colordict = {"frontal": "red", "frontocentral": "orange", "somatosensory": "yellow",
             "centroparietal": "green", "temporal": "blue", "parietal_occipital": "purple"}

CH_TO_GROUP = {ch: g for g, chs in groups.items() for ch in chs}


def initialize_graph(epochs):
    graph = nx.Graph()
    for i, ch in enumerate(epochs.ch_names):
        group = CH_TO_GROUP.get(ch, "")
        graph.add_node(i, color=colordict[group], name=ch, group=group)
    return graph


def build_corr_graph(epochs, mask, weights, threshold=0.2, use_abs=True):
    g = initialize_graph(epochs)
    mask = np.array(mask)
    weights = np.array(weights)
    n = len(epochs.ch_names)
    iu, ju = np.triu_indices(n, k=1)
    keep = mask[iu, ju] > threshold
    w = weights[iu[keep], ju[keep]]
    if use_abs:
        w = np.abs(w)
    g.add_weighted_edges_from(zip(iu[keep].tolist(), ju[keep].tolist(), w.tolist()))
    return g


In [ ]:
import csv
p1first_arr = np.genfromtxt('cisifirst10.csv', delimiter=',', skip_header=1)

seed = 245# int(np.random.random_integers(0,1000))
print(seed)
p1first10 = build_corr_graph(p1, mask = p1first_arr, weights = p1first_arr, threshold = 0.3)
plot_3d_graph(p1first10, seed)

In [ ]:
p1last_arr = np.genfromtxt('cisilast10.csv', delimiter=',', skip_header=1)
print(seed)
p1last10 = build_corr_graph(p1, mask = p1last_arr, weights = p1last_arr, threshold = 0.3)
plot_3d_graph(p1last10, seed)

In [ ]:
p2first_arr = np.genfromtxt('visifirst10 (2).csv', delimiter=',', skip_header=1)
print(seed)
p2first10 = build_corr_graph(p2, mask = p2first_arr, weights = p2first_arr, threshold = 0.3)
plot_3d_graph(p2first10,seed)

In [ ]:
p2last_arr = np.genfromtxt('visilast10 (2).csv', delimiter=',', skip_header=1)
print(seed)
p2last10 = build_corr_graph(p2, mask = p2last_arr, weights = p2last_arr, threshold = 0.3)
plot_3d_graph(p2last10, seed)

In [ ]:
clim = 1.0

fig, axes = plt.subplots(3, 2, figsize=(12, 8), constrained_layout=True)

# [0,0] Δ Corr averaged over all time at selected lag
im0 = axes[0, 0].imshow(p1first_arr, vmin=-clim, vmax=clim,
                                 cmap='RdBu_r', aspect='equal')
axes[0, 0].set_title(f'Corrected Correlation Matrix for P1 First 10 Trials')
fig.colorbar(im0, ax=axes[0, 0], label='Δ Pearson r', shrink=0.8)

im1 = axes[1, 0].imshow(p1last_arr, vmin=-clim, vmax=clim,
                                 cmap='RdBu_r', aspect='equal')
axes[1, 0].set_title(f'Corrected Correlation Matrix for P1 Last 10 Trials')
fig.colorbar(im1, ax=axes[1, 0], label='Δ Pearson r', shrink=0.8)

diff_p1 = np.subtract(p1last_arr, p1first_arr)
im2 = axes[2,0].imshow(diff_p1, vmin=-clim, vmax=clim, cmap = 'RdBu_r', aspect='equal')
axes[2,0].set_title('Difference between first and last 10 trials for P1')
fig.colorbar(im2, ax=axes[2,0], label='Δ Pearson r', shrink=0.8)

im3 = axes[0, 1].imshow(p2first_arr, vmin=-clim, vmax=clim,
                                 cmap='RdBu_r', aspect='equal')
axes[0, 1].set_title(f'Corrected Correlation Matrix for P2 First 10 Trials')
fig.colorbar(im3, ax=axes[0, 1], label='Δ Pearson r', shrink=0.8)

im4 = axes[1, 1].imshow(p2last_arr, vmin=-clim, vmax=clim,
                                 cmap='RdBu_r', aspect='equal')
axes[1, 1].set_title(f'Corrected Correlation Matrix for P2 Last 10 Trials')
fig.colorbar(im4, ax=axes[1, 1], label='Δ Pearson r', shrink=0.8)

diff_p2 = np.subtract(p2last_arr, p2first_arr)
im5 = axes[2,1].imshow(diff_p2, vmin=-clim, vmax=clim, cmap = 'RdBu_r', aspect='equal')
axes[2,1].set_title('Difference between first and last 10 trials for P2')
fig.colorbar(im5, ax=axes[2,1], label='Δ Pearson r', shrink=0.8)